## beamOptimizer > calc

In [ ]:
# before
class beamOptimizer:
    def calc(self, method, segmentVar, startPoint, objectives, plotProgress = False, plotBeam = False, printResults = False):
        '''
        optimizes beamline segment attribute values so y values are close to objective values as possible.
        Post optimization plotting supported

        Parameters
        ----------
        method: str
            name of minimization method/algorithm
        segmentVar: dict
            dictionary, each key is an indice corresponding to its value of a list of
            x variable parameters
        objectives: dict
            dictionary, each key is an indice corresponding to its value of a list of
            y objectives. In each list are dictionaries corresponding to the parameters of that
            y objective
        startPoint: dict
            dictionary, each key is an x variable corresponding to another dictionary of
            that variables' bounds, starting search point, and other parameters
        plotProgress: bool
            plot x variable and y objective values as a function of iterations
        plotBeam: bool
            plot beamline simulation with new x variables post-optimization
        printResults: bool
            output data in terminal

        Returns
        -------
        result: OptimizeResult
            Object containing resulting information about optimization process and results
        '''
        #  Variables for plotting purposes later
        self.plotMSE = []
        self.plotIterate = []
        self.trackVariables = []
        self.iterationTrack = 0
        self.trackGoals = {}

        #  Initialize set-list of x variables to optimize
        self.segmentVar = segmentVar
        checkSet = set()
        self.variablesToOptimize = []
        for indice in self.segmentVar:
            if (indice < 0 or indice >= len(self.beamline)):
                raise IndexError(str(indice) + " is out of bounds for segmentVar dictionary")
            checkSet.add(self.segmentVar.get(indice)[0])
        #  This entire program relies on checking the indice of the variables in this list-set.
        #  A little sketch, will work for now but better implementation is needed
        self.variablesToOptimize = list(checkSet)

        #  Initialize objectives dictionary with measurement methods
        self.objectives = objectives
        for key, value in self.objectives.items():
            if (key not in range(len(self.beamline))):
                raise TypeError("Invalid indice: indice " + str(key) + " in objectives dict is out of bounds" )
            for goal in value:
                if goal["measure"][1] in self.OBJECTIVEMETHODS:
                    goal["measure"][1] = self.OBJECTIVEMETHODS[goal["measure"][1]]
                elif isinstance(goal["measure"][1], str):
                    raise TypeError("Invalid method name: No such method name exists in OBJECTIVESMETHOD dict")
                #  Used to keep track of data plotting through optimization
                #  Very rudementary, since looking for plotting of an objective relies on finding the same string name. Will have to improve in future
                self.trackGoals.update({"indice " + str(key) + ": " + goal["measure"][0] + " "  + goal["measure"][1].__name__: []})

        #  Create x variables' bounds and  start point list.
        #  Order corresponding to x variable order in variablesToOptimize
        self.variablesValues = []
        self.bounds = []
        for i in self.variablesToOptimize:
            self.variablesValues.append(1)
            self.bounds.append((None, None))
        for var in startPoint:
            index = self.variablesToOptimize.index(var)
            if "start" in startPoint.get(var): self.variablesValues[index] = startPoint.get(var).get("start")
            if "bounds" in startPoint.get(var): self.bounds[index] = startPoint.get(var).get("bounds")

        # Time speed to minimize difference of objective function
        startTime = time.perf_counter()
        # print("degug beamOptimizer 194: ",self._optiSpeed, self.variablesValues)
        result = spo.minimize(self._optiSpeed, self.variablesValues, method=method, bounds=self.bounds)
        endTime = time.perf_counter()

        # print out new values for each beam segment's attribute
        output = "\nx variables:"
        for indice in self.segmentVar:
                variable = self.segmentVar.get(indice)[0]
                index = self.variablesToOptimize.index(variable)
                yFunc = self.segmentVar.get(indice)[2]
                newVal = yFunc(result.x[index])
                segAttr = self.segmentVar.get(indice)[1]
                setattr(self.beamline[indice], segAttr, newVal)
                if printResults:
                    output += "\nindice " + str(indice) + " new " + segAttr + " value: " + str(newVal)
        if printResults:
            output += "\n\ny objectives:\n"
            for indice, value in self.objectives.items():
                for obj in value:
                    output += "indice " + str(indice) + ": " + obj["measure"][0] + " "  + obj["measure"][1].__name__ + " value of " + str(obj["measured"]) + "\n"
            output += "Final difference: " + str(result.fun) + "\n"
            output += "\nTotal time: " + str(endTime-startTime) + " s\n"
            output +="Total iterations: " + str(self.iterationTrack) + "\n"
            print(output)

        # Plot the progress of y objectives and x variables as a function of iterations
        if plotProgress:
            fig, ax = plt.subplots(2,1)
            handles = []

            # plot MSE line
            mseLine, =ax[1].plot(self.plotIterate, self.plotMSE, label = 'Mean Squared Error', color = 'black')
            ax[1].set_xlabel('Iterations')
            ax[1].set_yscale('log')
            ax[1].set_ylabel('Mean Squared Error')
            ax[1].set_title("MSE and objectives vs Iterations")
            ax[1].tick_params(axis='y')
            handles.append(mseLine)

            # Plot y goals
            ax2 = ax[1].twinx()
            mini = 0
            for i, key in enumerate(self.trackGoals):
                valLine, = ax2.plot(self.plotIterate, self.trackGoals[key], label = key)
                handles.append(valLine)
                tempMin = abs(min(self.trackGoals[key]))
                if i == 0 or mini>tempMin:
                    mini = tempMin
            ax2.set_yscale('symlog', linthresh=10**(np.ceil(np.log10(mini))))
            ax2.set_ylabel('Objective functions')
            ax2.legend(handles = handles, loc = 'upper right')

            # Plot x variables + sec/iteration
            tempTrackVari = np.array(self.trackVariables)
            handles = []
            for i in range(len(tempTrackVari[0])):
                varLine, = ax[0].plot(self.plotIterate, tempTrackVari[:,i], label = self.variablesToOptimize[i])
                handles.append(varLine)
            ax[0].set_xlabel('Iterations')
            ax[0].set_ylabel('Variable values')
            ax[0].set_title('Variable Values through each Iteration')
            timeLine = mlines.Line2D([], [], color = 'white', label=f'{round((endTime-startTime)/self.iterationTrack,4)} s/iteration')
            handles.append(timeLine)
            ax[0].legend(handles = handles, loc = 'upper right')


            plt.tight_layout()
            plt.show()

        # Plot beam simulation with new values
        if plotBeam:
            schem = draw_beamline()
            tempPart = self.matrixVariables
            schem.plotBeamPositionTransform(tempPart, self.beamline)

        return result


In [ ]:
# after
class beamOptimizer:
    ...
    def calc(self, method, segmentVar, startPoint, objectives, plotProgress = False, plotBeam = False, printResults = False):
        '''
        optimizes beamline segment attribute values so y values are close to objective values as possible.
        Post optimization plotting supported

        Parameters
        ----------
        method: str
            name of minimization method/algorithm
        segmentVar: dict
            dictionary, each key is an indice corresponding to its value of a list of
            x variable parameters
        objectives: dict
            dictionary, each key is an indice corresponding to its value of a list of
            y objectives. In each list are dictionaries corresponding to the parameters of that
            y objective
        startPoint: dict
            dictionary, each key is an x variable corresponding to another dictionary of
            that variables' bounds, starting search point, and other parameters
        plotProgress: bool
            plot x variable and y objective values as a function of iterations
        plotBeam: bool
            plot beamline simulation with new x variables post-optimization
        printResults: bool
            output data in terminal

        Returns
        -------
        result: OptimizeResult
            Object containing resulting information about optimization process and results
        '''
        #  Variables for plotting purposes later
        self.plotMSE = []
        self.plotIterate = []
        self.trackVariables = []
        self.iterationTrack = 0
        self.trackGoals = {}

        #  Initialize set-list of x variables to optimize
        self.segmentVar = segmentVar
        checkSet = set()
        self.variablesToOptimize = []
        for indice in self.segmentVar:
            if (indice < 0 or indice >= len(self.beamline)):
                raise IndexError(str(indice) + " is out of bounds for segmentVar dictionary")
            checkSet.add(self.segmentVar.get(indice)[0])
        #  This entire program relies on checking the indice of the variables in this list-set.
        #  A little sketch, will work for now but better implementation is needed
        self.variablesToOptimize = list(checkSet)

        #  Initialize objectives dictionary with measurement methods
        self.objectives = objectives
        for key, value in self.objectives.items():
            if (key not in range(len(self.beamline))):
                raise TypeError("Invalid indice: indice " + str(key) + " in objectives dict is out of bounds" )
            for goal in value:
                if goal["measure"][1] in self.OBJECTIVEMETHODS:
                    goal["measure"][1] = self.OBJECTIVEMETHODS[goal["measure"][1]]
                elif isinstance(goal["measure"][1], str):
                    raise TypeError("Invalid method name: No such method name exists in OBJECTIVESMETHOD dict")
                #  Used to keep track of data plotting through optimization
                #  Very rudementary, since looking for plotting of an objective relies on finding the same string name. Will have to improve in future
                self.trackGoals.update({"indice " + str(key) + ": " + goal["measure"][0] + " "  + goal["measure"][1].__name__: []})

        #  Create x variables' bounds and  start point list.
        #  Order corresponding to x variable order in variablesToOptimize
        self.variablesValues = []
        self.bounds = []
        for i in self.variablesToOptimize:
            self.variablesValues.append(1)
            self.bounds.append((None, None))
        for var in startPoint:
            index = self.variablesToOptimize.index(var)
            if "start" in startPoint.get(var): self.variablesValues[index] = startPoint.get(var).get("start")
            if "bounds" in startPoint.get(var): self.bounds[index] = startPoint.get(var).get("bounds")

        # Time speed to minimize difference of objective function
        startTime = time.perf_counter()
        # print("debug beamOptimizer 194: ",self._optiSpeed, self.variablesValues)
        result = spo.minimize(self._optiSpeed, self.variablesValues, method=method, bounds=self.bounds)
        endTime = time.perf_counter()

        # print out new values for each beam segment's attribute
        output = "\nx variables:"
        for indice in self.segmentVar:
            variable = self.segmentVar.get(indice)[0]
            index = self.variablesToOptimize.index(variable)
            yFunc = self.segmentVar.get(indice)[2]
            newVal = yFunc(result.x[index])
            segAttr = self.segmentVar.get(indice)[1]
            setattr(self.beamline[indice], segAttr, newVal)
            if printResults:
                output += "\nindice " + str(indice) + " new " + segAttr + " value: " + str(newVal)
        if printResults:
            output += "\n\ny objectives:\n"
            for indice, value in self.objectives.items():
                for obj in value:
                    output += "indice " + str(indice) + ": " + obj["measure"][0] + " "  + obj["measure"][1].__name__ + " value of " + str(obj["measured"]) + "\n"
            output += "Final difference: " + str(result.fun) + "\n"
            output += "\nTotal time: " + str(endTime-startTime) + " s\n"
            output +="Total iterations: " + str(self.iterationTrack) + "\n"
            print(output)

        # Plot the progress of y objectives and x variables as a function of iterations
        if plotProgress:
            fig, ax = plt.subplots(2,1)
            handles = []

            # CRITICAL: Ensure self.plotIterate and self.plotMSE are NumPy compatible
            # If they were collected as Tensors, convert them here
            plot_iterate_np = np.array([i.item() if torch.is_tensor(i) else i for i in self.plotIterate])
            plot_mse_np = np.array([m.item() if torch.is_tensor(m) else m for m in self.plotMSE])

            # plot MSE line
            mseLine, = ax[1].plot(plot_iterate_np, plot_mse_np, label='Mean Squared Error', color='black')
            ax[1].set_xlabel('Iterations')
            ax[1].set_yscale('log')
            ax[1].set_ylabel('Mean Squared Error')
            ax[1].set_title("MSE and objectives vs Iterations")
            ax[1].tick_params(axis='y')
            handles.append(mseLine)

            # Plot y goals
            ax2 = ax[1].twinx()
            mini = 1e-6 # Default small value to avoid log(0)

            for i, key in enumerate(self.trackGoals):
                # CONVERSION: Convert the tracked goal list to a NumPy array for Matplotlib
                goal_data = np.array([g.item() if torch.is_tensor(g) else g for g in self.trackGoals[key]])

                valLine, = ax2.plot(plot_iterate_np, goal_data, label=key)
                handles.append(valLine)

                # Use numpy for absolute minimum calculation
                tempMin = np.abs(goal_data).min() if len(goal_data) > 0 else 1e-6
                if i == 0 or mini > tempMin:
                    mini = tempMin

            # FIX: Use a scalar float for linthresh, not a Tensor
            lin_thresh_val = 10**(np.ceil(np.log10(float(mini)))) if mini > 0 else 1e-6
            ax2.set_yscale('symlog', linthresh=lin_thresh_val)
            ax2.set_ylabel('Objective functions')
            ax2.legend(handles=handles, loc='upper right')

            # Plot x variables + sec/iteration
            # CONVERSION: Ensure trackVariables is a pure NumPy array
            tempTrackVari = np.array([[v.item() if torch.is_tensor(v) else v for v in row] for row in self.trackVariables])

            handles = []
            for i in range(len(tempTrackVari[0])):
                varLine, = ax[0].plot(plot_iterate_np, tempTrackVari[:,i], label=self.variablesToOptimize[i])
                handles.append(varLine)

            ax[0].set_xlabel('Iterations')
            ax[0].set_ylabel('Variable values')
            ax[0].set_title('Variable Values through each Iteration')

            calc_time = (endTime-startTime)/self.iterationTrack if self.iterationTrack > 0 else 0
            timeLine = mlines.Line2D([], [], color='white', label=f'{round(calc_time, 4)} s/iteration')
            handles.append(timeLine)
            ax[0].legend(handles=handles, loc='upper right')

            plt.tight_layout()
            plt.show()

        # Plot beam simulation with new values
        if plotBeam:
            schem = draw_beamline()
            tempPart = self.matrixVariables
            schem.plotBeamPositionTransform(tempPart, self.beamline)

        return result